# A tour: from a decision model to a ranking

This notebook walks the `decision_science` package once, from a YAML file to a ranked list
and three figures, over two small files committed under `tests/fixtures/tour/`. The numbers
are invented: four candidate sites for a regional supply depot, measured on four criteria.

Run the cells in order. Each code cell ends with `assert` lines, so a cell that finishes
without an error did what the text above it says. The repo's test suite runs every cell
the same way, so the tour cannot drift from the code.

To open it: `uv pip install --python .venv -e ".[decision-science,notebook]"`, then pick
the `.venv` interpreter as the kernel.

In [ ]:
import sys
from pathlib import Path

import yaml

ROOT = Path.cwd().parent  # the notebook runs from notebooks/
sys.path.insert(0, str(ROOT))

from src.myproject.decision_science import (
    MAUTScorer,
    dominance_check,
    monte_carlo,
    one_at_a_time,
    radar_chart,
    rank_stability_heatmap,
    robustness_report,
    scenario_compare,
    tornado_plot,
)

FIXTURES = ROOT / "tests" / "fixtures" / "tour"
MODEL = FIXTURES / "depot_model.yaml"
SITES = FIXTURES / "depot_sites.yaml"

assert MODEL.exists() and SITES.exists(), "run this notebook from the notebooks/ folder of the repo"

## 1. A model is a YAML file

A model lists criteria. Each criterion has a weight and a value function. The value
function turns a raw measurement (dollars, minutes, pallets) into a utility between 0 and
1, so that unlike units can be added. The weights say how much each criterion counts, and
they sum to 1.

In [ ]:
print(MODEL.read_text())

scorer = MAUTScorer.from_yaml(MODEL)
weights = {c.name: c.weight for c in scorer.criteria}

assert len(weights) == 4
assert abs(sum(weights.values()) - 1.0) < 0.01

## 2. A value function bends the scale

Two things to see in the table below.

`annual_cost` is a cost, so less is better. The model says so by giving `low: 900,
high: 300`: `low` is the raw value worth 0 and `high` the raw value worth 1, and nothing
requires `low` to be the smaller number.

`storage_pallets` is logarithmic: the step from 500 to 1,000 pallets is worth more than
the step from 4,000 to 4,500.

In [ ]:
PROBES = {
    "annual_cost": [900, 600, 300],
    "drive_minutes": [90, 50, 10],
    "storage_pallets": [500, 1000, 4000, 4500, 5000],
    "flood_free_years": [18, 24, 30],
}

value_fn = {c.name: c.value_fn for c in scorer.criteria}
for name, raws in PROBES.items():
    print(f"{name:18}", "  ".join(f"{raw:>5} -> {value_fn[name](raw):.2f}" for raw in raws))

assert value_fn["annual_cost"](300) == 1.0 and value_fn["annual_cost"](900) == 0.0
first_step = value_fn["storage_pallets"](1000) - value_fn["storage_pallets"](500)
late_step = value_fn["storage_pallets"](4500) - value_fn["storage_pallets"](4000)
assert first_step > 5 * late_step

## 3. Score one site

`score()` takes one alternative's raw measurements and returns its utility, with the
per-criterion parts kept. `explain()` gives those parts as a plain dict, largest first.

In [ ]:
sites = yaml.safe_load(SITES.read_text())["alternatives"]
print("raw:", sites["River Bend"])

river_bend = scorer.score("River Bend", sites["River Bend"])
explanation = river_bend.explain()
print(f"utility: {explanation['utility']}")
for part in explanation["criteria"]:
    print(f"  {part['name']:18} utility {part['raw_utility']:.2f}"
          f"  contributes {part['weighted_contribution']:.3f}  ({part['pct_of_total']:.0%} of the total)")

assert abs(sum(river_bend.breakdown.values()) - river_bend.utility) < 1e-9
assert explanation["criteria"][-1]["name"] == "flood_free_years"  # cheap and close, and it floods

## 4. Rank all four

`rank()` scores every alternative and sorts them. `dominance_check()` then finds any site
that another site beats or ties on every criterion; such a site cannot win under any
weights, so it can leave the discussion before anyone argues about weights.

In [ ]:
results = scorer.rank(sites)
for place, result in enumerate(results, start=1):
    print(f"{place}. {result.alternative:13} {result.utility:.3f}")

dominated = dominance_check(results)
for better, worse in dominated:
    print(f"\n{better} is at least as good as {worse} on every criterion")

assert [r.alternative for r in results][0] == "Airport Road"
assert dominated == [("Airport Road", "Old Mill")]

## 5. A figure: the shape of each site

The radar chart draws what each criterion contributes to each site's total: the
utility times the weight, so an axis reaches no further than its weight (0.30 for cost).
Airport Road has no weak axis; River Bend is the best on two axes and nearly empty on a
third.
The figure comes from the package (`radar_chart`), not from plotting code in this
notebook, so every notebook and report in the repo draws it the same way.

In [ ]:
radar = radar_chart(results, title="Four depot sites: what each criterion contributes to the total")

assert len(radar.axes[0].lines) == len(sites)

## 6. How sure is the ranking?

The weights are a judgment, so the honest question is whether the answer survives a
different judgment. Three tools, from narrow to wide.

**One weight at a time.** Move each weight up and down by 0.1, rescale the rest, and
rank again. The tornado plot shows how far the leader's utility moves for each criterion.

In [ ]:
oat = one_at_a_time(scorer, sites, delta=0.1)
leader = results[0].alternative

changes = {case: ranked[0].alternative for case, ranked in oat.items() if ranked[0].alternative != leader}
print("cases where the leader changes:", changes or "none")

tornado = tornado_plot(oat, leader, title=f"{leader}: utility when one weight moves by 0.1")

assert set(changes) == {"flood_free_years-delta"}

**Every weight at once.** `monte_carlo()` draws many weight vectors near the stated
ones and records how often each site lands in each place. The seed makes the draw
repeatable. The heatmap shows the result; `robustness_report()` reduces it to one line.

In [ ]:
frequencies = monte_carlo(scorer, sites, n_samples=2000, seed=7)
report = robustness_report(frequencies)

print(f"first most often: {report.winner} ({report.winner_frequency:.0%} of draws)")
print(f"then:             {report.runner_up} ({report.runner_up_frequency:.0%})")
print(f"robust:           {report.is_robust}")

heatmap = rank_stability_heatmap(frequencies, title="Share of 2,000 weight draws in which each site took each place")

for site, places in frequencies.items():
    assert abs(sum(places.values()) - 1.0) < 1e-9, site
assert frequencies["Old Mill"]["1"] == 0.0  # the dominated site never wins
assert not report.is_robust

Read the two results together. On the stated weights Airport Road leads. Across the
draws it is first or second almost every time, while River Bend is first about as often
and last about as often. That is a finding about the decision, not a fault in the tool:
the choice between them turns on how much flooding counts, and that is the weight to
settle with the people who own the decision.

**Named points of view.** `scenario_compare()` ranks the sites under whole weight sets
you name.

In [ ]:
SCENARIOS = {
    "cost first": {"annual_cost": 0.6, "drive_minutes": 0.2, "storage_pallets": 0.1, "flood_free_years": 0.1},
    "safety first": {"annual_cost": 0.1, "drive_minutes": 0.2, "storage_pallets": 0.1, "flood_free_years": 0.6},
}

by_scenario = scenario_compare(scorer, sites, SCENARIOS)
for name, ranked in by_scenario.items():
    print(f"{name:13}", " > ".join(r.alternative for r in ranked))

assert by_scenario["cost first"][0].alternative == "River Bend"
assert by_scenario["safety first"][-1].alternative == "River Bend"

## 7. Hands on: change it, break it, fix it

**Change it.** Edit `MY_WEIGHTS` below and run the cell. Try to find weights that put
Rail Yard first.

**Break it.** A weight set that does not sum to 1 is refused, and the message says
what the sum was. The second half of the cell shows the error you will meet most often.

In [ ]:
MY_WEIGHTS = {"annual_cost": 0.25, "drive_minutes": 0.25, "storage_pallets": 0.25, "flood_free_years": 0.25}

mine = scenario_compare(scorer, sites, {"mine": MY_WEIGHTS})["mine"]
print("mine:", " > ".join(f"{r.alternative} {r.utility:.3f}" for r in mine))

broken = dict(MY_WEIGHTS, annual_cost=0.5)  # now the four sum to 1.25
try:
    scenario_compare(scorer, sites, {"broken": broken})
except ValueError as error:
    message = str(error)
    print("refused:", message)
else:
    message = ""

assert len(mine) == len(sites)
assert message, "a weight set that sums to 1.25 was accepted"

**Other errors, and what they mean.**

| You see | Cause | Fix |
|---|---|---|
| `Weights must sum to 1.0` | the weights in the YAML or a scenario are off by more than 0.01 | rescale them |
| `Unknown value_fn` | a misspelled function name in the YAML | the message lists the seven valid names |
| `Invalid params for criterion` | a parameter the function does not take, such as `high` on `logistic` | check the function's signature in `value_functions.py` |
| `ImportError: matplotlib is required` | the figures' optional dependency is missing | run the install command the message prints |
| a site scores 0 or 1 on a criterion for every input | `low` and `high` do not span your data | print the value function over your range, as section 2 does |

## 8. Your own model

Point the two paths at your own files and run the cell. The sites file needs a top-level
`alternatives:` key, with each alternative giving a number for every criterion in the
model. Left as `None`, the cell does nothing.

Clear the outputs before you commit this notebook. Its outputs would carry your data,
and the test suite refuses a notebook that has any.

In [ ]:
MY_MODEL = None  # for example "~/decisions/vendor_model.yaml"
MY_ALTERNATIVES = None  # for example "~/decisions/vendors.yaml"

if MY_MODEL and MY_ALTERNATIVES:
    my_scorer = MAUTScorer.from_yaml(Path(MY_MODEL).expanduser())
    my_alternatives = yaml.safe_load(Path(MY_ALTERNATIVES).expanduser().read_text())["alternatives"]
    my_results = my_scorer.rank(my_alternatives)
    for place, result in enumerate(my_results, start=1):
        print(f"{place}. {result.alternative:20} {result.utility:.3f}")
    print(robustness_report(monte_carlo(my_scorer, my_alternatives, n_samples=2000, seed=7)))
    my_radar = radar_chart(my_results, title="My alternatives: what each criterion contributes to the total")

## Where next

- The seven value functions and their parameters: `src/myproject/decision_science/value_functions.py`
- Building a model from a table of elicited weights: `MAUTScorer.from_weights`
- What the tests hold for each function: `tests/unit/test_scorer.py`, `test_sensitivity.py`, `test_visualization.py`
- What this notebook must keep doing: `tests/unit/test_notebooks.py`
- `notebooks/cases_how_a_ranking_goes_wrong.ipynb`: four places where the obvious rule gives a different answer. It is a draft: a new feature, reviewed and still changing.